In [ ]:
import logging
import queue
import threading
import warnings
from pathlib import Path
from typing import Dict, Iterator, List, Optional, Tuple
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from shapely.geometry import Polygon
from shapely.errors import TopologicalError
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
except ImportError:
    CUDA_AVAILABLE = False
    warnings.warn("torch not found. Inference will run on CPU.", stacklevel=2)

from ultralytics import YOLO
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# TILE SAMPLER
class TilingSampler:
    """
    Generates (col_off, row_off, width, height) tile descriptors for a raster
    using a sliding window with configurable overlap.
    No image data is read here — purely geometric window arithmetic.
    """
 
    def __init__(
        self,
        raster_width:  int,    # Full raster width in pixels
        raster_height: int,    # Full raster height in pixels
        tile_size:     int,    # Tile side length in pixels (square)
        overlap:       int     # Overlap between adjacent tiles in pixels
    ) -> None:
        if overlap >= tile_size:
            raise ValueError(
                f"overlap ({overlap}) must be less than tile_size ({tile_size})."
            )
        self.raster_width  = raster_width
        self.raster_height = raster_height
        self.tile_size     = tile_size
        self.stride        = tile_size - overlap
 
    def __iter__(self) -> Iterator[Tuple[int, int, int, int]]:
        """Yields (col_off, row_off, tile_w, tile_h) for every valid tile."""
        for row_off in range(0, self.raster_height, self.stride):
            for col_off in range(0, self.raster_width, self.stride):
                tile_w = min(self.tile_size, self.raster_width  - col_off)
                tile_h = min(self.tile_size, self.raster_height - row_off)
                if tile_w >= 32 and tile_h >= 32:
                    yield col_off, row_off, tile_w, tile_h
 
    def __len__(self) -> int:
        cols = len(range(0, self.raster_width,  self.stride))
        rows = len(range(0, self.raster_height, self.stride))
        return cols * rows
 
# CROWN DETECTION ENGINE
class CrownDetectionEngine:
    """
    Producer-consumer GPU pipeline for parallel tree crown detection.
 
    Parameters
    ----------
    model_path      : Path to YOLOv11 .pt weights file.
    tile_size       : Inference tile size in pixels. Match model training size.
    overlap         : Tile overlap in pixels.
    conf_threshold  : YOLO confidence threshold [0-1].
    nms_iou_thresh  : IoU threshold for spatial duplicate suppression.
    min_diameter_m  : Minimum crown diameter filter in meters.
    n_readers       : Parallel raster reader threads. 4-8 recommended.
                      GPU inference always runs on a single dedicated thread.
    queue_size      : Max tiles buffered between readers and GPU thread.
    flush_every     : Polygon accumulation flush interval for RAM control.
    """
 
    def __init__(
        self,
        model_path:     str,
        tile_size:      int   = 960,
        overlap:        int   = 200,
        conf_threshold: float = 0.25,
        nms_iou_thresh: float = 0.50,
        min_diameter_m: float = 3.0,
        n_readers:      int   = 6,
        queue_size:     int   = 30,
        flush_every:    int   = 50
    ) -> None:
        self.model_path     = Path(model_path)
        self.tile_size      = tile_size
        self.overlap        = overlap
        self.conf_threshold = conf_threshold
        self.nms_iou_thresh = nms_iou_thresh
        self.min_diameter_m = min_diameter_m
        self.n_readers      = max(1, n_readers)
        self.queue_size     = queue_size
        self.flush_every    = flush_every
 
        # Auto-detect inference device
        if CUDA_AVAILABLE:
            self.device = f"cuda:{torch.cuda.device_count() - 1}"
        else:
            self.device = "cpu"
        logger.info(f"Inference device: {self.device}")
 
        # Load model ONCE — shared read-only across all threads
        logger.info(f"Loading YOLO model: {self.model_path.name}...")
        self.model = YOLO(str(self.model_path))
 
    # Internal helpers
    def _read_tile(
        self,
        src:     rasterio.DatasetReader,
        col_off: int,
        row_off: int,
        tile_w:  int,
        tile_h:  int
    ) -> np.ndarray:
        """Reads RGB bands for a tile. Returns HWC uint8 array."""
        window = Window(col_off, row_off, tile_w, tile_h)
        return np.moveaxis(src.read([1, 2, 3], window=window), 0, -1)
 
    def _infer_tile(
        self,
        tile_img: np.ndarray
    ) -> Optional[List[np.ndarray]]:
        """Runs YOLO segmentation on a tile. Returns mask list or None."""
        results = self.model.predict(
            tile_img,
            imgsz   = self.tile_size,
            conf    = self.conf_threshold,
            device  = self.device,
            half    = True,
            verbose = False
        )
        if results[0].masks is None:
            return None
        return [mask for mask in results[0].masks.xy if len(mask) >= 3]
 
    def _masks_to_polygons(
        self,
        masks:           List[np.ndarray],
        col_off:         int,
        row_off:         int,
        affine_transform
    ) -> List[Polygon]:
        """Converts tile-local pixel masks to georeferenced Polygons."""
        polygons: List[Polygon] = []
        for mask in masks:
            global_pixels = [(p[0] + col_off, p[1] + row_off) for p in mask]
            geo_coords    = [affine_transform * p for p in global_pixels]
            try:
                poly = Polygon(geo_coords).buffer(0)
                if poly.is_valid and not poly.is_empty:
                    polygons.append(poly)
            except (TopologicalError, ValueError):
                continue
        return polygons
 
    def _vectorized_nms(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        """
        Removes duplicate detections from overlapping tiles.
        Vectorized spatial join + IoU threshold — no Python for-loop.
        """
        if len(gdf) == 0:
            return gdf
 
        logger.info(f"Running vectorized NMS on {len(gdf):,} raw detections...")
 
        gdf = gdf.copy()
        gdf["_area"] = gdf.geometry.area
        gdf = gdf.sort_values("_area", ascending=False).reset_index(drop=True)
        gdf["_id"]   = gdf.index
 
        joined = gpd.sjoin(
            gdf[["_id", "_area", "geometry"]],
            gdf[["_id", "_area", "geometry"]],
            how="inner", predicate="intersects"
        )
        pairs = joined[joined["_id_left"] < joined["_id_right"]].copy()
 
        if len(pairs) > 0:
            def compute_iou(row) -> float:
                try:
                    gl    = gdf.loc[row["_id_left"],  "geometry"]
                    gr    = gdf.loc[row["_id_right"], "geometry"]
                    inter = gl.intersection(gr).area
                    union = gl.union(gr).area
                    return inter / union if union > 0 else 0.0
                except (TopologicalError, ValueError):
                    return 0.0
 
            pairs["_iou"] = pairs.apply(compute_iou, axis=1)
            suppress_ids  = set(
                pairs.loc[pairs["_iou"] > self.nms_iou_thresh, "_id_right"].tolist()
            )
        else:
            suppress_ids = set()
 
        gdf_clean = gdf[~gdf["_id"].isin(suppress_ids)].drop(
            columns=["_area", "_id"]
        ).reset_index(drop=True)
 
        logger.info(
            f"NMS: {len(gdf):,} raw → {len(gdf_clean):,} unique "
            f"(removed {len(gdf) - len(gdf_clean):,} duplicates)"
        )
        return gdf_clean
 
    # Producer-consumer workers
    def _reader_worker(
        self,
        image_path:   str,
        tile_queue:   queue.Queue,
        result_queue: queue.Queue
    ) -> None:
        """
        Reader thread: pops tile descriptors from tile_queue, reads the raster
        window, pushes (tile_img, col_off, row_off) to result_queue.
        Opens its own rasterio handle per call — avoids JPEG thread-safety issues.
        Stops on None poison pill.
        """
        while True:
            item = tile_queue.get()
            if item is None:
                tile_queue.task_done()
                result_queue.put(None)   # notify GPU thread this reader is done
                break
            col_off, row_off, tile_w, tile_h = item
            try:
                with rasterio.open(image_path) as src:
                    tile_img = self._read_tile(src, col_off, row_off, tile_w, tile_h)
                result_queue.put((tile_img, col_off, row_off))
            except Exception as e:
                logger.debug(f"Reader failed ({col_off},{row_off}): {e}")
            finally:
                tile_queue.task_done()
 
    def _gpu_inference_worker(
        self,
        result_queue:    queue.Queue,
        polygon_queue:   queue.Queue,
        affine_transform,
        n_readers:       int
    ) -> None:
        """
        GPU inference thread — exactly one instance.
        Drains result_queue, runs YOLO, pushes polygon lists to polygon_queue.
        Counts reader poison pills to know when all readers are done.
        """
        readers_done = 0
        tiles_done   = 0
 
        while True:
            item = result_queue.get()
 
            if item is None:
                readers_done += 1
                if readers_done >= n_readers:
                    polygon_queue.put(None)   # signal main thread to stop
                    break
                continue
 
            tile_img, col_off, row_off = item
            try:
                masks = self._infer_tile(tile_img)
                if masks:
                    polys = self._masks_to_polygons(masks, col_off, row_off, affine_transform)
                    if polys:
                        polygon_queue.put(polys)
            except Exception as e:
                logger.debug(f"GPU failed ({col_off},{row_off}): {e}")
 
            tiles_done += 1
            if tiles_done % 200 == 0:
                logger.info(f"  GPU: {tiles_done} tiles inferred...")
 
    # Main pipeline
    def detect(
        self,
        image_path: str
    ) -> Tuple[gpd.GeoDataFrame, Dict]:
        """
        Full detection pipeline on a large orthophoto.
 
        Steps:
            1. Read raster metadata only (no full image load).
            2. Generate tile windows via TilingSampler.
            3. Start N reader threads + 1 GPU thread in producer-consumer pattern.
            4. Main thread drains polygon_queue with periodic RAM flush.
            5. Global vectorized NMS.
            6. Diameter filter and attribute computation.
        """
        path = Path(image_path)
        if not path.exists():
            raise FileNotFoundError(f"Image not found: {path}")
 
        logger.info(f"Opening raster: {path.name}")
 
        # Step 1: metadata only
        with rasterio.open(str(path)) as src:
            affine        = src.transform
            crs           = src.crs
            width         = src.width
            height        = src.height
            res_x, res_y  = src.res
            scale         = 10
            data_mask     = src.dataset_mask(
                out_shape=(height // scale + 1, width // scale + 1)
            )
            valid_frac    = (data_mask > 0).mean()
            total_area_ha = (width * height * abs(res_x * res_y) * valid_frac) / 10000
 
        raster_meta = {
            "width":         width,
            "height":        height,
            "res_m":         abs(res_x),
            "crs":           crs,
            "total_area_ha": total_area_ha
        }
 
        logger.info(
            f"Raster: {width}x{height} px | GSD={abs(res_x)*100:.2f} cm/px | "
            f"Valid area={total_area_ha:.4f} ha"
        )
 
        # Step 2: tile windows
        sampler   = TilingSampler(width, height, self.tile_size, self.overlap)
        all_tiles = list(sampler)
        n_tiles   = len(all_tiles)
 
        logger.info(
            f"Tiling: {n_tiles} tiles | tile_size={self.tile_size} | "
            f"overlap={self.overlap} | readers={self.n_readers} | GPU=1 thread"
        )
 
        # Step 3: queues and threads
        tile_queue    = queue.Queue(maxsize=self.queue_size)
        result_queue  = queue.Queue(maxsize=self.queue_size)
        polygon_queue = queue.Queue()
 
        readers = []
        for _ in range(self.n_readers):
            t = threading.Thread(
                target=self._reader_worker,
                args=(image_path, tile_queue, result_queue),
                daemon=True
            )
            t.start()
            readers.append(t)
 
        gpu_thread = threading.Thread(
            target=self._gpu_inference_worker,
            args=(result_queue, polygon_queue, affine, self.n_readers),
            daemon=True
        )
        gpu_thread.start()
 
        # Enqueue all tiles then poison pills for readers
        for tile in all_tiles:
            tile_queue.put(tile)
        for _ in range(self.n_readers):
            tile_queue.put(None)
 
        # Step 4: drain polygon_queue in main thread
        all_polygons:    List[Polygon]          = []
        polygon_batches: List[gpd.GeoDataFrame] = []
        flush_count = 0
 
        while True:
            item = polygon_queue.get()
            if item is None:
                break
            all_polygons.extend(item)
            flush_count += 1
 
            if flush_count % self.flush_every == 0 and all_polygons:
                polygon_batches.append(
                    gpd.GeoDataFrame({"geometry": all_polygons}, crs=crs)
                )
                total_so_far = sum(len(b) for b in polygon_batches)
                logger.info(
                    f"  Flushed batch {len(polygon_batches)} | "
                    f"cumulative polygons: {total_so_far:,}"
                )
                all_polygons = []
 
        for t in readers:
            t.join()
        gpu_thread.join()
 
        # Flush remainder
        if all_polygons:
            polygon_batches.append(
                gpd.GeoDataFrame({"geometry": all_polygons}, crs=crs)
            )
 
        if not polygon_batches:
            logger.warning("No tree crowns detected.")
            return gpd.GeoDataFrame({"geometry": []}, crs=crs), raster_meta
 
        gdf_raw = gpd.GeoDataFrame(
            pd.concat(polygon_batches, ignore_index=True),
            geometry="geometry", crs=crs
        )
        logger.info(f"Total raw detections: {len(gdf_raw):,}")
 
        # Step 5: global vectorized NMS
        gdf_clean = self._vectorized_nms(gdf_raw)
 
        # Step 6: diameter filter
        gdf_clean["area_sqm"]     = gdf_clean.geometry.area.round(2)
        gdf_clean["crown_diam_m"] = (
            2.0 * np.sqrt(gdf_clean["area_sqm"] / np.pi)
        ).round(2)
        gdf_clean = gdf_clean[
            gdf_clean["crown_diam_m"] >= self.min_diameter_m
        ].copy().reset_index(drop=True)
 
        logger.info(
            f"After diameter filter (>={self.min_diameter_m}m): "
            f"{len(gdf_clean):,} crowns retained"
        )
 
        return gdf_clean, raster_meta
 
# INVENTORY REPORTER
class InventoryReporter:
    """Dasometric statistics and GeoPackage export."""
 
    @staticmethod
    def compute_statistics(
        gdf:         gpd.GeoDataFrame,
        raster_meta: Dict,
        output_path: str
    ) -> gpd.GeoDataFrame:
        """
        Adds ID and centroid coordinates, prints forest statistics,
        and exports the final GeoPackage.
        """
        if len(gdf) == 0:
            logger.warning("Empty GeoDataFrame — nothing to report.")
            return gdf
 
        gdf = gdf.copy().reset_index(drop=True)
        gdf["id"]      = range(1, len(gdf) + 1)
        centroids      = gdf.geometry.centroid
        gdf["coord_x"] = centroids.x.round(2)
        gdf["coord_y"] = centroids.y.round(2)
 
        total_area_ha = raster_meta["total_area_ha"]
        diameters     = gdf["crown_diam_m"]
        canopy_cover  = (gdf["area_sqm"].sum() / (total_area_ha * 10_000)) * 100
        density       = len(gdf) / total_area_ha
 
        print("\n" + "=" * 50)
        print("       FOREST INVENTORY STATISTICS")
        print("=" * 50)
        print(f"  GSD                  : {raster_meta['res_m']*100:.2f} cm/px")
        print(f"  Valid area analyzed  : {total_area_ha:.4f} ha")
        print(f"  Detected stems       : {len(gdf):,} trees")
        print(f"  Density              : {density:.2f} stems/ha")
        print(f"  Canopy Cover (FCC)   : {canopy_cover:.2f} %")
        print("-" * 50)
        print(f"  Mean crown diameter  : {diameters.mean():.2f} m")
        print(f"  Std deviation (diam) : {diameters.std():.2f} m")
        print(f"  Min diameter         : {diameters.min():.2f} m")
        print(f"  Max diameter         : {diameters.max():.2f} m")
        print(f"  Median diameter      : {diameters.median():.2f} m")
        print("=" * 50)
 
        out_path = Path(output_path)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        gdf[["id", "area_sqm", "crown_diam_m", "coord_x", "coord_y", "geometry"]].to_file(
            str(out_path), driver="GPKG"
        )
        logger.info(f"GeoPackage exported: {out_path.name}")
        return gdf
 
 
# ENTRY POINT
if __name__ == "__main__":
 
    IMAGE_PATH  = "./Proyecto_Copas/Ortofoto_jarales.tif"
    MODEL_PATH  = "./Proyecto_Copas/models/best.pt"
    OUTPUT_GPKG = "./Forest_Inventory_Results.gpkg"
 
    engine = CrownDetectionEngine(
        model_path     = MODEL_PATH,
        tile_size      = 960,     # Match YOLO training imgsz
        overlap        = 200,     # Tile overlap in pixels
        conf_threshold = 0.20,    # YOLO confidence threshold
        nms_iou_thresh = 0.50,    # Duplicate suppression IoU threshold
        min_diameter_m = 3.0,     # Minimum crown diameter in meters
        n_readers      = 6,       # Parallel raster reader threads (keep GPU fed) (Change as your GPU needs)
        queue_size     = 30,      # Tile buffer between readers and GPU
        flush_every    = 50       # Polygon flush interval (RAM control)
    )
 
    gdf_crowns, meta = engine.detect(IMAGE_PATH)
 
    InventoryReporter.compute_statistics(
        gdf         = gdf_crowns,
        raster_meta = meta,
        output_path = OUTPUT_GPKG
    )
